<a href="https://colab.research.google.com/github/watermelonripeness-thesis/Watermelon-Audio-Model/blob/main/esp32_export_no_undersample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ESP32-S3 Export & Compatibility Check (No-Undersample Model)

Trains embedded-friendly candidates on the **no-undersample** feature set
(`train_features_no_undersample.csv` / `test_features_no_undersample.csv`),
exports each to C via `micromlgen`, then compiles each against the
**actual ESP32-S3 WROOM CAM N16R8** target (16MB flash, 8MB octal-SPI
PSRAM) to get real flash/RAM numbers -- not estimates.

**Candidates in this notebook:** Naive Bayes, Decision Tree, Random
Forest (capped tree counts: 10/25/50/100), Logistic Regression, and
Linear SVM.

**Note on Linear SVM:** its exported header size scales with the
number of support vectors, which depends entirely on how separable the
data turns out to be -- it can end up anywhere from a few KB to
several hundred KB. Check its row in the size comparison table below
before assuming it's a viable embedded candidate; a linear decision
boundary doesn't guarantee a small model.

**Excluded, and why:**
- **XGBoost** -- confirmed non-exportable via both `micromlgen` and
  `m2cgen` (model structure isn't supported by either).
- **KNN** -- technically exportable, but needs the entire training set
  stored on-device plus a distance calculation per prediction; not
  practical for embedded.
- **RBF / poly / sigmoid-kernel SVM** -- exportable, but every
  prediction needs a kernel evaluation against every support vector,
  which is expensive on-device compared to a linear decision boundary.
  Only the **linear** kernel is included here.

**Class imbalance note:** the underlying data is NOT undersampled
(271 unripe / 124 ripe in train), so `class_weight="balanced"` is
included in the search space for every model that supports it (all
except Naive Bayes, which doesn't have that parameter).

**Confound caveat:** the accuracy/F1 numbers below are evaluated on the
same confounded dataset as the full model comparison -- they're useful
for picking which model compiles smallest and cheapest on real
hardware, but shouldn't be read as "this model detects ripeness" until
data recollection is done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install micromlgen --quiet


  Preparing metadata (setup.py) ... done


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from micromlgen import port

RANDOM_STATE = 42
DATA_DIR = "/content/drive/MyDrive/Watermelon Data/z. csv"
OUT_DIR = "/content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample"
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(RANDOM_STATE)


## Load features (no-undersample run)

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_features_no_undersample.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test_features_no_undersample.csv"))

FEATURE_COLS = [c for c in train_df.columns if c not in ("filename", "label")]

Xtr = train_df[FEATURE_COLS].values
ytr = (train_df["label"] == "ripe").astype(int).values
Xte = test_df[FEATURE_COLS].values
yte = (test_df["label"] == "ripe").astype(int).values

print(f"Train: {len(Xtr)} samples ({train_df['label'].value_counts().to_dict()})")
print(f"Test:  {len(Xte)} samples ({test_df['label'].value_counts().to_dict()})")
print(f"Features: {len(FEATURE_COLS)}")


Train: 395 samples ({'unripe': 271, 'ripe': 124})
Test:  99 samples ({'unripe': 68, 'ripe': 31})
Features: 31


## Train embedded-friendly candidates

`GridSearchCV`, 5-fold stratified CV, scored on F1 (not accuracy --
matters given the class imbalance).


In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
candidates = {}

# --- Decision Tree ---
dt_grid = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"],
    "class_weight": [None, "balanced"],
}
dt_search = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), dt_grid, cv=CV, scoring="f1")
dt_search.fit(Xtr, ytr)
candidates["Decision Tree"] = dt_search.best_estimator_

# --- Random Forest, capped tree counts (flash size scales ~linearly with tree count) ---
for n_trees in [10, 25, 50, 100]:
    rf_grid = {
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "class_weight": [None, "balanced"],
    }
    rf_search = GridSearchCV(
        RandomForestClassifier(n_estimators=n_trees, random_state=RANDOM_STATE),
        rf_grid, cv=CV, scoring="f1",
    )
    rf_search.fit(Xtr, ytr)
    candidates[f"Random Forest ({n_trees} trees)"] = rf_search.best_estimator_

# --- Naive Bayes ---
nb_grid = {"var_smoothing": np.logspace(-11, -7, 30)}
nb_search = GridSearchCV(GaussianNB(), nb_grid, cv=CV, scoring="f1")
nb_search.fit(Xtr, ytr)
nb_best = nb_search.best_estimator_
# sklearn renamed sigma_ -> var_ in newer versions; micromlgen still expects sigma_
if not hasattr(nb_best, "sigma_") and hasattr(nb_best, "var_"):
    nb_best.sigma_ = nb_best.var_
candidates["Naive Bayes"] = nb_best

# --- Logistic Regression ---
lr_grid = {
    "C": np.logspace(-3, 2, 15),
    "penalty": ["l2"],
    "solver": ["lbfgs"],
    "class_weight": [None, "balanced"],
}
lr_search = GridSearchCV(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE), lr_grid, cv=CV, scoring="f1")
lr_search.fit(Xtr, ytr)
candidates["Logistic Regression"] = lr_search.best_estimator_

# --- Linear SVM (kernel fixed to "linear" -- see exclusions above) ---
svm_grid = {
    "C": np.logspace(-2, 2, 15),
    "class_weight": [None, "balanced"],
}
svm_search = GridSearchCV(
    SVC(kernel="linear", gamma=0.001, random_state=RANDOM_STATE),  # gamma unused by
    svm_grid, cv=CV, scoring="f1",                                # linear kernel, but
)                                                                   # micromlgen requires
                                                                     # an explicit float
svm_search.fit(Xtr, ytr)
candidates["Linear SVM"] = svm_search.best_estimator_

print("Trained candidates:", list(candidates.keys()))


Trained candidates: ['Decision Tree', 'Random Forest (10 trees)', 'Random Forest (25 trees)', 'Random Forest (50 trees)', 'Random Forest (100 trees)', 'Naive Bayes', 'Logistic Regression', 'Linear SVM']


## Evaluate accuracy on held-out test set

In [ ]:
eval_rows = []
for name, model in candidates.items():
    pred = model.predict(Xte)
    eval_rows.append({
        "Model": name,
        "Test Accuracy": round(accuracy_score(yte, pred), 4),
        "Test Precision": round(precision_score(yte, pred, zero_division=0), 4),
        "Test Recall": round(recall_score(yte, pred, zero_division=0), 4),
        "Test F1": round(f1_score(yte, pred, zero_division=0), 4),
    })
eval_df = pd.DataFrame(eval_rows)
print(eval_df.to_string(index=False))


                    Model  Test Accuracy  Test Precision  Test Recall  Test F1
            Decision Tree         0.9899          1.0000       0.9677   0.9836
 Random Forest (10 trees)         0.9899          1.0000       0.9677   0.9836
 Random Forest (25 trees)         0.9798          1.0000       0.9355   0.9667
 Random Forest (50 trees)         0.9697          0.9667       0.9355   0.9508
Random Forest (100 trees)         0.9697          0.9667       0.9355   0.9508
              Naive Bayes         0.9596          0.9355       0.9355   0.9355
      Logistic Regression         0.9798          1.0000       0.9355   0.9667
               Linear SVM         0.9798          1.0000       0.9355   0.9667


## Export each to C code and measure generated header size

In [ ]:
size_rows = []
generated_code = {}

for name, model in candidates.items():
    try:
        c_code = port(model)
        size_kb = len(c_code.encode("utf-8")) / 1024
        generated_code[name] = c_code
        size_rows.append({"Model": name, "Header Size (KB)": round(size_kb, 2), "Export": "OK"})
    except Exception as e:
        size_rows.append({"Model": name, "Header Size (KB)": None, "Export": f"FAILED: {type(e).__name__}"})

size_df = pd.DataFrame(size_rows)
print(size_df.to_string(index=False))


                    Model  Header Size (KB) Export
            Decision Tree              1.67     OK
 Random Forest (10 trees)             27.20     OK
 Random Forest (25 trees)             73.48     OK
 Random Forest (50 trees)            104.82     OK
Random Forest (100 trees)            269.34     OK
              Naive Bayes              5.04     OK
      Logistic Regression              2.00     OK
               Linear SVM             31.72     OK


## Combined comparison: accuracy vs. footprint

In [ ]:
combined = eval_df.merge(size_df, on="Model")
combined = combined.sort_values("Test F1", ascending=False).reset_index(drop=True)
print(combined.to_string(index=False))


                    Model  Test Accuracy  Test Precision  Test Recall  Test F1  Header Size (KB) Export
            Decision Tree         0.9899          1.0000       0.9677   0.9836              1.67     OK
 Random Forest (10 trees)         0.9899          1.0000       0.9677   0.9836             27.20     OK
 Random Forest (25 trees)         0.9798          1.0000       0.9355   0.9667             73.48     OK
               Linear SVM         0.9798          1.0000       0.9355   0.9667             31.72     OK
      Logistic Regression         0.9798          1.0000       0.9355   0.9667              2.00     OK
 Random Forest (50 trees)         0.9697          0.9667       0.9355   0.9508            104.82     OK
Random Forest (100 trees)         0.9697          0.9667       0.9355   0.9508            269.34     OK
              Naive Bayes         0.9596          0.9355       0.9355   0.9355              5.04     OK


## Save generated C headers

In [ ]:
for name, c_code in generated_code.items():
    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    out_path = os.path.join(OUT_DIR, f"{safe_name}.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    print(f"Saved {out_path}")

combined.to_csv(os.path.join(OUT_DIR, "esp32_model_comparison_no_undersample.csv"), index=False)
print(f"\nSaved comparison table to {os.path.join(OUT_DIR, 'esp32_model_comparison_no_undersample.csv')}")


Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/decision_tree.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/random_forest_10_trees.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/random_forest_25_trees.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/random_forest_50_trees.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/random_forest_100_trees.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/naive_bayes.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/logistic_regression.h
Saved /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/linear_svm.h

Saved comparison table to /content/drive/MyDrive/Watermelon Data/esp32_models_no_undersample/esp32_model_comparison_no_undersample.csv


---
## Real compiled flash/RAM check -- ESP32-S3 WROOM CAM (N16R8)

Compiling does **not** require the physical board to be connected --
only uploading does. This gives real, exact flash/RAM numbers straight
from the compiler, not estimates.


In [ ]:
!curl -fsSL https://raw.githubusercontent.com/arduino/arduino-cli/master/install.sh | sh
!/content/bin/arduino-cli version


Installing in /content/bin
ARCH=64bit
OS=Linux
Using curl as download tool
install.sh: arduino-cli not found. You might want to add "/content/bin" to your $PATH
arduino-cli  Version: 1.5.1 Commit: 01f3d4f2b Date: 2026-06-05T10:22:17Z installed successfully in /content/bin
arduino-cli  Version: 1.5.1 Commit: 01f3d4f2b Date: 2026-06-05T10:22:17Z


In [ ]:
import shutil

candidate_paths = [
    shutil.which("arduino-cli"),
    "/content/bin/arduino-cli",
    os.path.expanduser("~/bin/arduino-cli"),
    "/root/bin/arduino-cli",
    "/usr/local/bin/arduino-cli",
]
ARDUINO_CLI = next((c for c in candidate_paths if c and os.path.exists(c)), None)
print(f"Using arduino-cli at: {ARDUINO_CLI}")
assert ARDUINO_CLI, "arduino-cli not found -- re-run the install cell above."


Using arduino-cli at: /content/bin/arduino-cli


## Install the ESP32 board core

Downloads the ESP32 toolchain (~1GB) the first time -- that's expected,
not an error.


In [ ]:
!{ARDUINO_CLI} config init --overwrite
!{ARDUINO_CLI} config add board_manager.additional_urls https://raw.githubusercontent.com/espressif/arduino-esp32/gh-pages/package_esp32_index.json
!{ARDUINO_CLI} core update-index
!{ARDUINO_CLI} core install esp32:esp32


Config file written to: /root/.arduino15/arduino-cli.yaml
builtin:ctags@5.8-arduino11 downloaded
Installing builtin:ctags@5.8-arduino11...
Skipping tool configuration....
builtin:ctags@5.8-arduino11 installed
builtin:dfu-discovery@0.1.2 downloaded
Installing builtin:dfu-discovery@0.1.2...
Skipping tool configuration....
builtin:dfu-discovery@0.1.2 installed
builtin:mdns-discovery@1.1.0 downloaded
Installing builtin:mdns-discovery@1.1.0...
Skipping tool configuration....
builtin:mdns-discovery@1.1.0 installed
builtin:serial-discovery@1.5.2 downloaded
Installing builtin:serial-discovery@1.5.2...
Skipping tool configuration....
builtin:serial-discovery@1.5.2 installed
builtin:serial-monitor@0.15.0 downloaded
Installing builtin:serial-monitor@0.15.0...
Skipping tool configuration....
builtin:serial-monitor@0.15.0 installed
esp32:esp-x32@2601 downloaded
esp32:xtensa-esp-elf-gdb@17.1_20260402 downloaded
esp32:esp-rv32@2601 downloaded
esp32:riscv32-esp-elf-gdb@17.1_20260402 downloaded
esp32:o

## Board configuration -- ESP32-S3 WROOM CAM (N16R8)

N16R8 = 16MB flash ("N16"), 8MB octal-SPI PSRAM ("R8"), per the
thesis's hardware spec. If this exact option string isn't accepted by
the installed core version, the fallback (plain `esp32s3` FQBN with
default options) is tried automatically.


In [ ]:
BOARD_FQBN_PRIMARY = "esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi"
BOARD_FQBN_FALLBACK = "esp32:esp32:esp32s3"


## Model -> (header file, generated class name) mapping

Matches the file names and class names produced by `micromlgen` above.


In [ ]:
MODEL_MAP = {
    "Decision Tree": ("decision_tree.h", "DecisionTree"),
    "Random Forest (10 trees)": ("random_forest_10_trees.h", "RandomForest"),
    "Random Forest (25 trees)": ("random_forest_25_trees.h", "RandomForest"),
    "Random Forest (50 trees)": ("random_forest_50_trees.h", "RandomForest"),
    "Random Forest (100 trees)": ("random_forest_100_trees.h", "RandomForest"),
    "Naive Bayes": ("naive_bayes.h", "GaussianNB"),
    "Logistic Regression": ("logistic_regression.h", "LogisticRegression"),
    "Linear SVM": ("linear_svm.h", "SVM"),
}

# Only attempt to compile models that actually exported successfully above
MODEL_MAP = {name: v for name, v in MODEL_MAP.items() if name in generated_code}

HEADER_DIR = OUT_DIR
SKETCH_ROOT = "/content/sketches"
print("Will compile-check:", list(MODEL_MAP.keys()))


Will compile-check: ['Decision Tree', 'Random Forest (10 trees)', 'Random Forest (25 trees)', 'Random Forest (50 trees)', 'Random Forest (100 trees)', 'Naive Bayes', 'Logistic Regression', 'Linear SVM']


## Build and compile a minimal sketch per model

In [ ]:
import re
import subprocess

os.makedirs(SKETCH_ROOT, exist_ok=True)

SKETCH_TEMPLATE = """#include "__HEADER_FILE__"
using namespace Eloquent::ML::Port;

__CLASS_NAME__ clf;

void setup() {
    Serial.begin(115200);
    float x[__N_FEATURES__] = {__DUMMY_VALUES__};  // dummy input, only to
                                                     // force the compiler to
                                                     // include the full
                                                     // predict() path
    int result = clf.predict(x);
    Serial.println(result);
}

void loop() {
}
"""


def build_sketch(header_file, class_name, n_features, dummy_values):
    return (SKETCH_TEMPLATE
            .replace("__HEADER_FILE__", header_file)
            .replace("__CLASS_NAME__", class_name)
            .replace("__N_FEATURES__", str(n_features))
            .replace("__DUMMY_VALUES__", dummy_values))


def parse_size_output(output):
    """Extract flash and RAM usage from arduino-cli compile output."""
    flash_match = re.search(r"Sketch uses (\d+) bytes.*?of program storage", output, re.DOTALL)
    ram_match = re.search(r"Global variables use (\d+) bytes.*?of dynamic memory", output, re.DOTALL)
    flash_bytes = int(flash_match.group(1)) if flash_match else None
    ram_bytes = int(ram_match.group(1)) if ram_match else None
    return flash_bytes, ram_bytes


def compile_model(name, header_file, class_name, n_features):
    sketch_dir = os.path.join(SKETCH_ROOT, name.lower().replace(" ", "_").replace("(", "").replace(")", ""))
    os.makedirs(sketch_dir, exist_ok=True)
    sketch_path = os.path.join(sketch_dir, os.path.basename(sketch_dir) + ".ino")

    shutil.copy(os.path.join(HEADER_DIR, header_file), os.path.join(sketch_dir, header_file))

    dummy_values = ", ".join(["0.1f"] * n_features)
    sketch_code = build_sketch(header_file, class_name, n_features, dummy_values)
    with open(sketch_path, "w") as f:
        f.write(sketch_code)

    for fqbn in (BOARD_FQBN_PRIMARY, BOARD_FQBN_FALLBACK):
        result = subprocess.run(
            [ARDUINO_CLI, "compile", "--fqbn", fqbn, sketch_dir],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            flash_bytes, ram_bytes = parse_size_output(result.stdout)
            return {
                "Model": name, "FQBN used": fqbn, "Compile": "OK",
                "Flash (bytes)": flash_bytes, "Flash (KB)": round(flash_bytes / 1024, 2) if flash_bytes else None,
                "RAM (bytes)": ram_bytes,
            }
    return {
        "Model": name, "FQBN used": None, "Compile": "FAILED",
        "Flash (bytes)": None, "Flash (KB)": None, "RAM (bytes)": None,
    }


n_features = len(FEATURE_COLS)
compile_rows = []
for name, (header_file, class_name) in MODEL_MAP.items():
    print(f"Compiling {name} ...")
    compile_rows.append(compile_model(name, header_file, class_name, n_features))

compile_df = pd.DataFrame(compile_rows)
print(compile_df.to_string(index=False))


Compiling Decision Tree ...
Compiling Random Forest (10 trees) ...
Compiling Random Forest (25 trees) ...
Compiling Random Forest (50 trees) ...
Compiling Random Forest (100 trees) ...
Compiling Naive Bayes ...
Compiling Logistic Regression ...
Compiling Linear SVM ...
                    Model                                   FQBN used Compile  Flash (bytes)  Flash (KB)  RAM (bytes)
            Decision Tree esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
 Random Forest (10 trees) esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
 Random Forest (25 trees) esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
 Random Forest (50 trees) esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
Random Forest (100 trees) esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         304079      296.95        22068
              Naive Ba

## Final comparison -- accuracy, F1, header size, and REAL compiled flash/RAM

In [ ]:
final = combined.merge(compile_df, on="Model", how="left")
final = final.sort_values("Test F1", ascending=False).reset_index(drop=True)
pd.set_option("display.width", 160)
print(final.to_string(index=False))

final.to_csv(os.path.join(OUT_DIR, "esp32_final_comparison_no_undersample.csv"), index=False)
print(f"\nSaved to {os.path.join(OUT_DIR, 'esp32_final_comparison_no_undersample.csv')}")


                    Model  Test Accuracy  Test Precision  Test Recall  Test F1  Header Size (KB) Export                                   FQBN used Compile  Flash (bytes)  Flash (KB)  RAM (bytes)
            Decision Tree         0.9899          1.0000       0.9677   0.9836              1.67     OK esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
 Random Forest (10 trees)         0.9899          1.0000       0.9677   0.9836             27.20     OK esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
 Random Forest (25 trees)         0.9798          1.0000       0.9355   0.9667             73.48     OK esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         278743      272.21        22068
               Linear SVM         0.9798          1.0000       0.9355   0.9667             31.72     OK esp32:esp32:esp32s3:FlashSize=16M,PSRAM=opi      OK         309135      301.89        22068
      Logistic Regre

## Reading this table

- **Flash (KB)** is what actually matters for the N16R8's 16MB flash --
  plenty of headroom either way, but smaller is still preferable for
  OTA update margin and boot time.
- **RAM (bytes)** matters more given the 8MB PSRAM is mostly for
  camera/frame buffers, not general heap -- keep this low.
- Cross-reference this against the **Test F1** column, but remember the
  confound caveat from the top of this notebook: don't pick a model
  based on F1 alone until data recollection is done. Right now, use
  this table primarily to confirm every embedded-friendly candidate
  **fits comfortably** on the ESP32-S3 -- pick the actual deployment
  model once the classifier is trained on clean, recollected data.
